# Where a random surfer spends its time

A surfer moves around a small web of pages. At each step, with probability $d = 0.8$, it
clicks one of the links on its current page, each equally likely. Otherwise it types in the
address of a page chosen uniformly at random from all $n$ pages, and it does the same
whenever its page has no links at all. The long-run share of time it spends on each page is
a way to score the pages: a page scores high when well-scored pages link to it.

The shares form the stationary distribution of a Markov chain, the eigenvector of its
one-step matrix with eigenvalue 1. You will build the chain, run it until it settles, and
then predict from the eigenvalues how long that takes.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# links[i] lists the pages that page i links to. Page 4 links nowhere.
links = {0: [1, 2], 1: [2], 2: [0], 3: [0, 2, 4], 4: [], 5: [3, 4]}
n = 6

## Part 1 (coding task): Where a click leads

Write `link_matrix(links, n)`, the $n \times n$ matrix $M$ whose column $i$ holds the
probabilities of where the surfer is next when it is on page $i$ and does not type in an
address: the click step, together with what happens on a page without links. Store the
probability of moving from page $i$ to page $j$ in `M[j, i]`, so that a column vector of
probabilities $\mathbf{r}$ becomes $M\mathbf{r}$.

In [ ]:
def link_matrix(links, n):
    # TODO: column i is the surfer's next-page distribution from page i.
    raise NotImplementedError

In [ ]:
M = np.asarray(link_matrix(links, n), dtype=float)
assert M.shape == (6, 6), f"expected shape (6, 6), got {M.shape}"
assert np.allclose(M.sum(axis=0), 1), f"every column is a probability distribution and should add up to 1, got {np.round(M.sum(axis=0), 4)}"
assert np.isclose(M[1, 0], 0.5) and np.isclose(M[0, 1], 0.0), "M[j, i] is the move from page i to page j: page 0 links to page 1, so M[1, 0] = 0.5, and page 1 does not link to page 0"
assert np.allclose(M[:, 4], 1 / 6), "page 4 has no links, so from there the surfer goes to any of the 6 pages"
assert np.isclose(M[4, 3], 1 / 3) and np.isclose(M[3, 5], 0.5), "page 3 has three links and page 5 has two"
print("Part 1 passed")

## Part 2 (coding task): Run the chain until it settles

Write `surfer_scores(M, d=0.8, tol=1e-10, max_steps=10_000)`. The surfer starts on page 0.
Track the vector $\mathbf{r}$ of probabilities of being on each page, apply one step of the
surfer's rule at a time, and stop at the first step after which no probability has changed by
more than `tol`. Return `(r, steps)`, the vector and the number of steps taken.

In [ ]:
def surfer_scores(M, d=0.8, tol=1e-10, max_steps=10_000):
    M = np.asarray(M, dtype=float)
    # TODO: one step of the surfer's rule at a time, from page 0, until nothing moves by more than tol.
    raise NotImplementedError

In [ ]:
M = np.asarray(link_matrix(links, n), dtype=float)
r, steps = surfer_scores(M)
r = np.asarray(r, dtype=float)
assert r.shape == (6,), f"expected one score per page, shape (6,), got {r.shape}"
assert np.isclose(r.sum(), 1), f"the scores are probabilities and should add up to 1, got {r.sum():.6f}"
G = 0.8 * M + 0.2 / 6                    # one step as a matrix, for r that adds up to 1
vals, vecs = np.linalg.eig(G)
top = vecs[:, np.argmin(np.abs(vals - 1))].real
assert np.allclose(r, top / top.sum(), atol=1e-8), "r is not the stationary distribution of the surfer's chain: a step clicks with probability d and types a uniform address with probability 1 - d"
assert 30 <= steps <= 50, f"from page 0 the chain needs about 40 steps to settle to 1e-10, got {steps}: count the steps, and stop when the largest change is at most tol"
r_half, _ = surfer_scores(M, d=0.5)
assert np.allclose(0.5 * M @ r_half + 0.5 / 6, r_half, atol=1e-8), "surfer_scores ignores its d argument"
print("scores:", np.round(r, 4), " steps:", steps)
print("ranking, best first:", [int(i) for i in np.argsort(-r)])
print("Part 2 passed")

## Part 3 (coding task): Predict the number of steps

Write `predict_steps(M, d=0.8, tol=1e-10)`, which predicts the step count of
`surfer_scores` without running its loop. Use the eigenvalues of the one-step matrix $G$ for
the rate at which the changes shrink. You may take a single step from the start to measure
the size of the first change. Return a whole number.

In [ ]:
def predict_steps(M, d=0.8, tol=1e-10):
    M = np.asarray(M, dtype=float)
    # TODO: the rate from the eigenvalues of G, the first change from one step, then solve for the count.
    raise NotImplementedError

In [ ]:
M = np.asarray(link_matrix(links, n), dtype=float)
pred = predict_steps(M)
_, actual = surfer_scores(M)
assert isinstance(pred, (int, np.integer)), f"return a whole number of steps, got {pred!r}"
assert abs(pred - actual) <= 3, f"predicted {pred} steps, but surfer_scores took {actual}: the changes shrink by the second-largest absolute eigenvalue of G = d M + (1 - d)/n per step, not by d or by an eigenvalue of M"
pred_half, (_, actual_half) = predict_steps(M, d=0.5), surfer_scores(M, d=0.5)
assert abs(pred_half - actual_half) <= 3, f"with d = 0.5 the prediction is {pred_half} against {actual_half} steps"
lam = np.sort(np.abs(np.linalg.eigvals(0.8 * M + 0.2 / 6)))[::-1]
print("|eigenvalues| of G:", np.round(lam, 4))
print(f"predicted {pred} steps, took {actual}")
print("Part 3 passed")

## Solution

### Part 1: Where a click leads

In [ ]:
def link_matrix(links, n):
    M = np.zeros((n, n))
    for i in range(n):
        targets = links.get(i, [])
        if targets:
            M[targets, i] = 1.0 / len(targets)   # each link on page i equally likely
        else:
            M[:, i] = 1.0 / n                    # no links: any page, uniformly
    return M

**Why it works.** 
Column $i$ is the surfer's next-page distribution from page $i$, so it must add up to 1.
Filling rows instead gives $M^\top$, which moves probability the wrong way. A page with no
links would leave its column at zero, and the chain would lose that page's share at every
step: the total would drift below 1. Sending the surfer anywhere from such a page keeps $M$ a
proper Markov matrix, which is what the description says it does.

### Part 2: Run the chain until it settles

In [ ]:
def surfer_scores(M, d=0.8, tol=1e-10, max_steps=10_000):
    M = np.asarray(M, dtype=float)
    n = M.shape[0]
    r = np.zeros(n)
    r[0] = 1.0
    for step in range(1, max_steps + 1):
        r_new = d * (M @ r) + (1 - d) / n        # click with probability d, else type an address
        if np.abs(r_new - r).max() <= tol:
            return r_new, step
        r = r_new
    return r, max_steps

**Why it works.** 
With probability $d$ the surfer clicks, which moves the probabilities by $M$, and with
probability $1 - d$ it types an address, which spreads that share evenly: one step is
$\mathbf{r} \leftarrow dM\mathbf{r} + \tfrac{1 - d}{n}\mathbf{1}$. Since $\mathbf{r}$ always adds
up to 1, the step is the matrix $G = dM + \tfrac{1 - d}{n}\mathbf{1}\mathbf{1}^\top$ applied to
$\mathbf{r}$, a Markov matrix with every entry positive, and repeating it is power iteration on
$G$. Its fixed point is the eigenvector with eigenvalue 1. Rescaling $\mathbf{r}$ each step
instead of adding the typed-address share runs power iteration on $M$ alone, which finds a
different vector. Page 2 ranks first because pages 0, 1 and 3 all link to it.

### Part 3: Predict the number of steps

In [ ]:
def predict_steps(M, d=0.8, tol=1e-10):
    M = np.asarray(M, dtype=float)
    n = M.shape[0]
    G = d * M + (1 - d) / n
    rate = np.sort(np.abs(np.linalg.eigvals(G)))[-2]   # second-largest |eigenvalue|
    r0 = np.zeros(n)
    r0[0] = 1.0
    first = np.abs(G @ r0 - r0).max()                  # size of the change on step 1
    return int(np.ceil(1 + np.log(tol / first) / np.log(rate)))

**Why it works.** 
Write the start as the stationary vector plus parts along the other eigenvectors of $G$. Each
step multiplies the part along an eigenvector by its eigenvalue, so after a few steps the
change is dominated by the second-largest $|\lambda|$, here about 0.566, and shrinks by that
factor per step. Starting from the first change $c_1$, the change after step $k$ is about
$c_1|\lambda_2|^{k-1}$, which drops below `tol` when
$k \ge 1 + \log(\text{tol}/c_1)/\log|\lambda_2|$, about 42 steps. The eigenvalues of $G$ other
than 1 are $d$ times those of $M$ (the uniform part only touches the eigenvalue 1), so
$|\lambda_2| \le d$ always: the jump is what guarantees a rate. Using $d$ itself, or the largest eigenvalue of $M$ other than 1, predicts
 too many steps, and a smaller eigenvalue of $M$ predicts too few. The second eigenvalue here is complex, so the
changes shrink in an uneven pattern, but on average by $|\lambda_2|$.
